# 04 · Select the held-out evaluation sample
**Paper:** Section II-A, Section III

The pipeline is evaluated on a **separate sample of 1,000 impure images**. The sample is
stratified by dominant impurity and excludes the 1,000 annotated images and all
near-duplicates. We also draw **200 clean images**, which are merged unchanged into the final
recovered dataset (Step 7).

The samples used in the paper are provided in `data/evaluation/`.

**Output** → `outputs/evaluation/test_dirty_sample.csv`, `outputs/evaluation/test_clean_sample.csv`

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import pandas as pd

from requilt.data import ANY_IMPAIRMENT, MULTI_PANEL, add_dominant_impurity, read_file_list, stratified_sample

THRESHOLD = cfg.impurity.threshold
out_dir = P.output_dir / "evaluation"
out_dir.mkdir(exist_ok=True)

## 1 · Exclude annotated images and near-duplicates

In [ ]:
scores = pd.read_csv(P.output_dir / "quilt_impurity_scores.csv")
annotated = set(read_file_list(P.annotation_dir / "train.txt"))
near_dups = set(pd.read_csv(P.output_dir / "near_duplicate_exclusions.csv")["Filename"])

available = scores[~scores["Filename"].isin(annotated | near_dups)].copy()
available["is_dirty"] = available[ANY_IMPAIRMENT] >= THRESHOLD
print(f"images available after exclusions: {len(available):,}")

## 2 · 1,000 impure images, stratified by dominant impurity

In [ ]:
dirty_pool = add_dominant_impurity(available[available["is_dirty"]])
test_dirty = stratified_sample(dirty_pool, cfg.evaluation.n_dirty, seed=cfg.seed)

print(test_dirty["dominant_impurity"].value_counts().to_string())
n_mp = int((test_dirty[MULTI_PANEL] >= THRESHOLD).sum())
print(f"\nimpure sample: {len(test_dirty)} images, {n_mp} flagged multi-panel")

## 3 · 200 clean images

In [ ]:
clean_pool = available[~available["is_dirty"]]
test_clean = clean_pool.sample(n=cfg.evaluation.n_clean, random_state=cfg.seed).reset_index(drop=True)
print(f"clean sample: {len(test_clean)} images")

## 4 · Save

In [ ]:
test_dirty.to_csv(out_dir / "test_dirty_sample.csv", index=False)
test_clean.to_csv(out_dir / "test_clean_sample.csv", index=False)
print(f"saved -> {out_dir}")